# FedUCA Extra-A2: Missing Experiments

Completes what Extra-A didn't finish:
1. Extended seeds (6-9) for PGA vs FedProx at alpha=0.1
2. K=20 alpha=0.1 FedAvg and PGA
3. 5-fold lesion-grouped cross-validation

**Add data:** `ham10000-vit-embeddings-orig` | **GPU T4** | Internet OFF

In [ ]:
import os, json, time, copy, glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_extra_a2"
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
NUM_CLASSES = 7
EMBED_DIM   = 768

hits = glob.glob("/kaggle/input/**/embeddings_train.npz", recursive=True)
if not hits:
    raise FileNotFoundError("Add ham10000-vit-embeddings-orig dataset first.")
EMBED_DIR = os.path.dirname(hits[0])
print(f"Device    : {DEVICE}")
print(f"Embeddings: {EMBED_DIR}")

In [ ]:
# Load embeddings
def load_split(name):
    d = np.load(f"{EMBED_DIR}/embeddings_{name}.npz", allow_pickle=True)
    embs = torch.tensor(d['embeddings'], dtype=torch.float32)
    lbls = torch.tensor(d['labels'],     dtype=torch.long)
    ids  = d['lesion_ids']
    return embs, lbls, ids

emb_train, lbl_train, ids_train = load_split("train")
emb_val,   lbl_val,   ids_val   = load_split("val")
emb_test,  lbl_test,  ids_test  = load_split("test")
emb_tv  = torch.cat([emb_train, emb_val],  dim=0)
lbl_tv  = torch.cat([lbl_train, lbl_val],  dim=0)
ids_tv  = np.concatenate([ids_train, ids_val])

print(f"Train:{emb_train.shape} Val:{emb_val.shape} Test:{emb_test.shape}")
counts  = torch.bincount(lbl_train, minlength=NUM_CLASSES).float()
weights = (1.0/counts); weights = (weights/weights.mean()).to(DEVICE)
print(f"Class weights: {weights.cpu().numpy().round(3)}")

In [ ]:
# All utilities
class BottleneckMLP(nn.Module):
    def __init__(self, ratio=4):
        super().__init__()
        b = 768 // ratio
        self.net = nn.Sequential(
            nn.Linear(768,4*b),nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,b),  nn.GELU(),nn.LayerNorm(b),  nn.Dropout(0.2),
            nn.Linear(b,4*b),  nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,7))
    def forward(self, x): return self.net(x)

def dirichlet_partition(labels, K, alpha, seed):
    rng = np.random.default_rng(seed)
    idx = [[] for _ in range(K)]
    for c in range(int(labels.max())+1):
        ci = np.where(labels.numpy()==c)[0]; rng.shuffle(ci)
        props  = rng.dirichlet(np.full(K, alpha))
        splits = (np.cumsum(props)*len(ci)).astype(int)[:-1]
        for k, chunk in enumerate(np.split(ci, splits)):
            idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

@torch.no_grad()
def evaluate(head, embs, labels):
    head.eval(); p, t = [], []
    for x, y in DataLoader(TensorDataset(embs, labels), batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu()); t.append(y)
    p=torch.cat(p).numpy(); t=torch.cat(t).numpy()
    f1s=[]
    for c in range(NUM_CLASSES):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9); f1s.append(2*pr*rc/(pr+rc+1e-9))
    return {"accuracy":float((p==t).mean()),"macro_f1":float(np.mean(f1s)),
            "f1_per_class":[float(x) for x in f1s]}

def fedavg(states, sizes):
    total=sum(sizes)
    return {k:sum((sizes[i]/total)*states[i][k].float() for i in range(len(sizes)))
            for k in states[0]}

def local_train_fedavg(head, embs, labels, lr=5e-3, epochs=5, bs=32):
    h=copy.deepcopy(head).to(DEVICE); h.train()
    opt=optim.Adam(h.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        for x,y in DataLoader(TensorDataset(embs.to(DEVICE),labels.to(DEVICE)),
                               batch_size=bs,shuffle=True):
            opt.zero_grad(); crit(h(x),y).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def local_train_fedprox(head, embs, labels, global_state,
                         mu=0.01, lr=5e-3, epochs=5, bs=32):
    h=copy.deepcopy(head).to(DEVICE); h.train()
    opt=optim.Adam(h.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=weights)
    gp={k:v.to(DEVICE) for k,v in global_state.items()}
    for _ in range(epochs):
        for x,y in DataLoader(TensorDataset(embs.to(DEVICE),labels.to(DEVICE)),
                               batch_size=bs,shuffle=True):
            opt.zero_grad()
            prox=torch.tensor(0.,device=DEVICE)
            for name,param in h.named_parameters():
                if name in gp: prox+=(param-gp[name]).pow(2).sum()
            (crit(h(x),y)+(0.01/2)*prox).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def compute_prototypes(embs, labels):
    proto=torch.zeros(NUM_CLASSES,EMBED_DIM)
    count=torch.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        mask=labels==c
        if mask.sum()>0:
            proto[c]=embs[mask].mean(0); count[c]=float(mask.sum())
    return proto,count

def prototype_guided_aggregation(states, sizes, client_protos, beta=5.0):
    K=len(states); total=sum(sizes)
    g_proto=torch.zeros(NUM_CLASSES,EMBED_DIM)
    g_count=torch.zeros(NUM_CLASSES)
    for k in range(K):
        pk,ck=client_protos[k]
        for c in range(NUM_CLASSES):
            if ck[c]>0: g_proto[c]+=ck[c]*pk[c]; g_count[c]+=ck[c]
    for c in range(NUM_CLASSES):
        if g_count[c]>0: g_proto[c]/=g_count[c]
    cos_d=lambda a,b: 1.0-float((a/(a.norm()+1e-9))@(b/(b.norm()+1e-9)))
    w=torch.zeros(NUM_CLASSES,K)
    for c in range(NUM_CLASSES):
        if g_proto[c].norm()<1e-9:
            for k in range(K): w[c,k]=sizes[k]/total
        else:
            lw=torch.zeros(K)
            for k in range(K):
                pk,ck=client_protos[k]
                lw[k]=-beta*cos_d(pk[c],g_proto[c]) if ck[c]>0 else -1e6
            w[c]=torch.softmax(lw,dim=0)
    fw_key=fb_key=None
    for key in states[0]:
        s=states[0][key].shape
        if len(s)==2 and s[0]==NUM_CLASSES: fw_key=key
        if len(s)==1 and s[0]==NUM_CLASSES: fb_key=key
    avg={}
    for key in states[0]:
        if key==fw_key:
            avg[key]=torch.stack([sum(w[c,k]*states[k][key][c].float()
                                      for k in range(K)) for c in range(NUM_CLASSES)])
        elif key==fb_key:
            avg[key]=torch.stack([sum(w[c,k]*states[k][key][c].float()
                                      for k in range(K)) for c in range(NUM_CLASSES)])
        else:
            avg[key]=sum((sizes[i]/total)*states[i][key].float() for i in range(K))
    return avg

def run_method(method, K, alpha, seed, n_rounds=60, epochs=5, lr=5e-3, bs=32):
    partition=dirichlet_partition(lbl_train,K,alpha,seed)
    c_embs  =[emb_train[idx] for idx in partition]
    c_labels=[lbl_train[idx] for idx in partition]
    sizes   =[len(idx) for idx in partition]
    if method=='pga':
        c_protos=[compute_prototypes(c_embs[k],c_labels[k]) for k in range(K)]
    head=BottleneckMLP(ratio=4).to(DEVICE)
    best_val=-1; best_round=0; best_state=None
    for rnd in range(1,n_rounds+1):
        gs=copy.deepcopy(head.state_dict())
        states=[]
        for k in range(K):
            if method=='fedprox':
                st=local_train_fedprox(head,c_embs[k],c_labels[k],gs,
                                        0.01,lr,epochs,bs)
            else:
                st=local_train_fedavg(head,c_embs[k],c_labels[k],lr,epochs,bs)
            states.append(st)
        if method=='pga':
            head.load_state_dict(prototype_guided_aggregation(states,sizes,c_protos))
        else:
            head.load_state_dict(fedavg(states,sizes))
        vm=evaluate(head,emb_val.to(DEVICE),lbl_val)
        if vm['macro_f1']>best_val:
            best_val=vm['macro_f1']; best_round=rnd
            best_state=copy.deepcopy(head.state_dict())
    head.load_state_dict(best_state)
    tm=evaluate(head,emb_test.to(DEVICE),lbl_test)
    return {"method":method,"K":K,"alpha":alpha,"seed":seed,
            "best_round":best_round,"best_val_f1":best_val,"test":tm}

print("All utilities ready.")

In [ ]:
# ── PART 1: Extended seeds 5-9 for PGA and FedProx at alpha=0.1 ──────
# This increases statistical power for the PGA vs FedProx comparison
print("="*60)
print("PART 1: Extended seeds (5-9) — PGA and FedProx at alpha=0.1")
print("="*60)

results_ext = []
EXTRA_SEEDS = [5, 6, 7, 8, 9]

for method in ['pga', 'fedprox', 'fedavg']:
    print(f"\n--- {method} alpha=0.1 seeds 5-9 ---")
    for seed in EXTRA_SEEDS:
        res = run_method(method, K=4, alpha=0.1, seed=seed)
        results_ext.append(res)
        with open(os.path.join(RESULTS_DIR,
                  f"{method}_k4_a0.1_s{seed}.json"),"w") as f:
            json.dump(res,f,indent=2)
        print(f"  {method} seed={seed}: F1={res['test']['macro_f1']:.3f}")
    f1s=[r['test']['macro_f1'] for r in results_ext
         if r['method']==method and r['alpha']==0.1]
    print(f"  {method} seeds 5-9 mean: {np.mean(f1s):.3f}±{np.std(f1s):.3f}")

print("\nPart 1 complete.")

In [ ]:
# ── PART 2: K=20 alpha=0.1 (missing from Extra-A) ───────────────────
print("="*60)
print("PART 2: K=20 clients at alpha=0.1 (missing from Extra-A)")
print("="*60)

results_k20 = []
for method in ['fedavg','pga']:
    print(f"\n--- {method} K=20 alpha=0.1 ---")
    for seed in range(5):
        res = run_method(method, K=20, alpha=0.1, seed=seed)
        results_k20.append(res)
        with open(os.path.join(RESULTS_DIR,
                  f"{method}_k20_a0.1_s{seed}.json"),"w") as f:
            json.dump(res,f,indent=2)
        print(f"  {method} K=20 seed={seed}: F1={res['test']['macro_f1']:.3f}")
    f1s=[r['test']['macro_f1'] for r in results_k20 if r['method']==method]
    print(f"  {method} K=20 alpha=0.1: {np.mean(f1s):.3f}±{np.std(f1s):.3f}")

print("\nPart 2 complete.")

In [ ]:
# ── PART 3: 5-fold lesion-grouped cross-validation ───────────────────
# FIX A: CV test score = held-out fold score (not fixed test set).
#   For each fold, the held-out fold is the validation; we train on the
#   remaining folds and evaluate on the held-out fold. The fixed test set
#   is NOT touched during CV. Its evaluation is reported separately.
# FIX B: Fold-specific class weights are computed once per fold and not
#   modified between methods within the same fold.

print("=" * 60)
print("PART 3: 5-fold Lesion-Grouped Cross-Validation (K=4, alpha=1000)")
print("=" * 60)

results_cv = []
gkf    = GroupKFold(n_splits=5)
splits = list(gkf.split(np.arange(len(emb_tv)), lbl_tv.numpy(), groups=ids_tv))

for fold, (tr_idx, va_idx) in enumerate(splits):
    print(f"\n--- Fold {fold+1}/5 ---")
    fold_emb_tr = emb_tv[tr_idx]; fold_lbl_tr = lbl_tv[tr_idx]
    fold_emb_va = emb_tv[va_idx]; fold_lbl_va = lbl_tv[va_idx]

    # Fold-specific class weights — computed once, shared across both methods
    fc = torch.bincount(fold_lbl_tr, minlength=NUM_CLASSES).float()
    fc = torch.clamp(fc, min=1)   # avoid division by zero for empty classes
    fw = (1.0 / fc); fw = (fw / fw.mean()).to(DEVICE)

    for method in ["fedavg", "pga"]:
        partition = dirichlet_partition(fold_lbl_tr, 4, 1000, seed=fold)
        c_embs    = [fold_emb_tr[idx] for idx in partition]
        c_labels  = [fold_lbl_tr[idx] for idx in partition]
        sizes     = [len(idx) for idx in partition]
        if method == "pga":
            c_protos = [compute_prototypes(c_embs[k], c_labels[k])
                        for k in range(4)]

        head = BottleneckMLP(ratio=4).to(DEVICE)
        best_val = -1; best_state = None

        for rnd in range(1, 61):
            # Pass fold-specific weights to local training
            states = [local_train_fedavg(head, c_embs[k], c_labels[k],
                                         class_weights=fw)
                      for k in range(4)]
            if method == "pga":
                head.load_state_dict(
                    prototype_guided_aggregation(states, sizes, c_protos))
            else:
                head.load_state_dict(fedavg(states, sizes))

            # FIX A: Use held-out fold (fold_emb_va) for model selection
            vm = evaluate(head, fold_emb_va.to(DEVICE), fold_lbl_va)
            if vm["macro_f1"] > best_val:
                best_val  = vm["macro_f1"]
                best_state = copy.deepcopy(head.state_dict())

        # FIX A: Report held-out fold score as the CV test score
        head.load_state_dict(best_state)
        fold_test_m = evaluate(head, fold_emb_va.to(DEVICE), fold_lbl_va)

        # Also compute score on the fixed primary test set for reference only
        fixed_test_m = evaluate(head, emb_test.to(DEVICE), lbl_test)

        res = {
            "method": method, "fold": fold,
            "val_f1": best_val,
            "out_of_fold_f1": fold_test_m["macro_f1"],  # the CV estimate
            "fixed_test": fixed_test_m,                 # reference only
            "test": fold_test_m                         # CV-reported metric
        }
        results_cv.append(res)
        print(f"  {method} fold {fold+1}: val={best_val:.3f} "
              f"out-of-fold={fold_test_m['macro_f1']:.3f} "
              f"(fixed_test={fixed_test_m['macro_f1']:.3f} — reference)")

with open(os.path.join(RESULTS_DIR, "cv_results.json"), "w") as f:
    json.dump(results_cv, f, indent=2)
print("\nPart 3 complete. 'test' key = out-of-fold score (CV estimate).")
print("'fixed_test' key = primary test-set score (reference, not CV estimate).")


In [ ]:
# ── FINAL SUMMARY ────────────────────────────────────────────────────
print("\n" + "="*60)
print("COMBINED RESULTS SUMMARY")
print("="*60)

# Load original seeds 0-4 from extra-a if available
ea_hits = glob.glob("/kaggle/input/**/fedprox_k4_a0.1_s*.json", recursive=True)
orig_fp = [json.load(open(f))['test']['macro_f1'] for f in ea_hits]
orig_pg_hits=glob.glob("/kaggle/input/**/pga_a0.1_s*.json",recursive=True)
orig_pg=[json.load(open(f))['test']['macro_f1'] for f in orig_pg_hits if '_s' in f]

ext_fp=[r['test']['macro_f1'] for r in results_ext
        if r['method']=='fedprox' and r['alpha']==0.1]
ext_pg=[r['test']['macro_f1'] for r in results_ext
        if r['method']=='pga'     and r['alpha']==0.1]
ext_fa=[r['test']['macro_f1'] for r in results_ext
        if r['method']=='fedavg'  and r['alpha']==0.1]

all_fp = orig_fp + ext_fp
all_pg = orig_pg + ext_pg

print(f"\nPGA vs FedProx at alpha=0.1 (10 seeds combined):")
print(f"  PGA    : {np.mean(all_pg):.3f} ± {np.std(all_pg):.3f} (n={len(all_pg)})")
print(f"  FedProx: {np.mean(all_fp):.3f} ± {np.std(all_fp):.3f} (n={len(all_fp)})")
if len(all_pg)==len(all_fp) and len(all_pg)>=5:
    _,pv=ttest_rel(all_pg[:min(len(all_pg),len(all_fp))],
                   all_fp[:min(len(all_pg),len(all_fp))])
    delta=np.mean(all_pg)-np.mean(all_fp)
    print(f"  Delta={delta:+.4f} | p={pv:.4f} | {'SIGNIFICANT' if pv<0.05 else 'ns'}")

print(f"\n5-fold CV (alpha=1000):")
for method in ['fedavg','pga']:
    rows=[r for r in results_cv if r['method']==method]
    if rows:
        label="FedAvg" if method=='fedavg' else "PGA"
        vf1s=[r['val_f1']           for r in rows]
        tf1s=[r['test']['macro_f1'] for r in rows]
        print(f"  {label}: val={np.mean(vf1s):.3f}±{np.std(vf1s):.3f} "
              f"test={np.mean(tf1s):.3f}±{np.std(tf1s):.3f}")

print(f"\nFiles saved to: {RESULTS_DIR}")
print("Save output as dataset: feduca-extra-a2-results")